# Informe de evaluación de athenis-voces

Regenera la tabla de la sección 12.1 del funcional a partir de las salidas guardadas, **sin volver a procesar el audio** (FR-015, RNF-07). Usa un solo script, `backend/src/evaluacion/evaluar.py`.

Solo mide la partición de prueba: EN2002a, EN2002c, ES2004a, IS1009a y TS3003a de AMI, y las grabaciones anotadas. Nada de lo que salga aquí se usa para ajustar reglas ni umbrales (constitución, regla 9).

## Antes de ejecutarlo

1. Descargar los datos: `scripts/descargar_datos.sh`.
2. Analizar las 5 reuniones de prueba con `backend/scripts/analizar_audio.py`, dejando las salidas **fuera del repositorio** (por defecto, `~/salidas`).
3. Ejecutar las dos celdas de código.

Las carpetas se pueden cambiar con las variables de entorno `SALIDAS`, `AMI` y `ETIQUETAS`. El cuaderno se guarda sin resultados: las cifras salen al ejecutarlo.

In [ ]:
import os
import subprocess
from pathlib import Path

# Las salidas quedan fuera del repositorio (RD-04)
SALIDAS = Path(os.environ.get("SALIDAS", "~/salidas")).expanduser()
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "backend" / "pyproject.toml").exists())
AMI = Path(os.environ.get("AMI", RAIZ / "datos" / "ami" / "anotaciones_but" / "only_words" / "rttms" / "test"))
ETIQUETAS = Path(os.environ.get("ETIQUETAS", RAIZ / "evaluacion" / "etiquetas"))

In [ ]:
# El mismo comando que en la terminal, con el entorno del proyecto (uv), aunque Jupyter use otro
orden = [
    "uv", "run", "--group", "evaluacion", "python", "-m", "evaluacion.evaluar",
    "--salidas", str(SALIDAS), "--ami", str(AMI), "--etiquetas", str(ETIQUETAS),
]
proceso = subprocess.run(
    orden,
    cwd=RAIZ / "backend",
    env={**os.environ, "PYTHONPATH": "src"},
    capture_output=True,
    text=True,
    check=False,
)
if proceso.returncode != 0:
    raise RuntimeError(proceso.stderr)
tabla = proceso.stdout

try:
    from IPython.display import Markdown, display

    display(Markdown(tabla))
except ImportError:
    print(tabla)

## Cómo leer la tabla

- **Cada cifra lleva su intervalo de confianza del 95 % y su n.** Con tan pocos datos, los resultados son orientativos y no se generalizan a otros episodios (RN-12, 11.3).
- **«No aplicable»** significa que no se puede calcular, por ejemplo porque faltan las anotaciones; nunca es un 0 (RN-13).
- **M-02 y M-03, la voz:** la unidad es el hablante y el intervalo es de Wilson (D-21). Acertar todas las voces no prueba que el modelo nunca falle: el intervalo dice hasta dónde podría fallar.
- **Las voces de AMI las etiqueta el modelo**, sin una persona que confirme (`analizar_audio.py`). En la aplicación, una persona confirma cada voz (regla 8).
- **La ablación** compara las interrupciones con la separación de voces de pyannote y con la de referencia de AMI. Cuanto más baja, más cambian las interrupciones por los errores al separar voces: es el límite principal del sistema.
- **M-05 y M-08** necesitan las anotaciones de las dos anotadoras y la referencia acordada (`<audio>_referencia.txt`).